# KKR Performance Analysis: IPL 2023–2024

## 1. Objective

Analyse Kolkata Knight Riders' performance across IPL 2023 and IPL 2024
to identify team performance trends, player contributions, and
potential tactical insights.

## 2. Analytical Scope

- Team-level performance and match outcomes
- Batting contributions and scoring efficiency
- Bowling contributions and effectiveness
- Performance across Powerplay, Middle, and Death phases
- Evidence-based scouting and tactical recommendations

## 3. Data Limitations

- The source dataset contains delivery-level records.
- KKR's final two league matches in IPL 2024 are absent.
- IPL 2024 Qualifier 1 and Final are included.
- Previously identified bowling-run anomalies remain unresolved.

Conclusions must account for data coverage, player roles, and the
limitations of the available metrics. Observed associations must not
be presented as proof of causation.

In [7]:
from pathlib import Path
import sys
import pandas as pd

current_path = Path.cwd()

project_root = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "src" / "config.py").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing src/config.py."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print("Project root successfully added to the Python path.")

Project root: /Users/soham/MyDocs/Projects/Cricket Analysis/Hackathon - DPL 2026/CodeFiles/Hackathon_DPL2026
Project root successfully added to the Python path.


In [3]:
from src.data_loader import load_raw_data
from src.features.metric_definitions import add_metric_features
from src.features.match_level import (
    build_match_summary,
    build_team_match,
)
from src.features.player_level import (
    build_player_season_batting,
    build_player_season_bowling,
    build_player_season_phase_batting,
    build_player_season_phase_bowling,
)

print("All required project modules imported successfully.")

All required project modules imported successfully.


In [4]:
df = load_raw_data()

print(f"Raw dataset shape: {df.shape}")
display(df.head())

Raw dataset shape: (34967, 61)


,row_id,p_match,inns,bat,p_bat,team_bat,bowl,p_bowl,team_bowl,ball,...,wagonzone,line,length,shot,control,predscore,wprob,bowl_speed_category,bowl_release_point,bowl_type
0,225793,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,1,...,6,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,162,49.42,3.0,2.0,OUTSWINGER
1,225794,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,2,...,0,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,FLICK,0.0,161,48.27,3.0,2.0,OUTSWINGER
2,225795,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,3,...,5,OUTSIDE_OFFSTUMP,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,160,47.12,2.0,2.0,ANGLED IN
3,225796,1359475,1,Ruturaj Gaikwad,1060380,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,4,...,4,ON_THE_STUMPS,GOOD_LENGTH,ON_DRIVE,1.0,160,47.12,2.0,2.0,OUTSWINGER
4,225797,1359475,1,Devon Conway,379140,Chennai Super Kings,Mohammed Shami,481896,Gujarat Titans,5,...,5,ON_THE_STUMPS,SHORT_OF_A_GOOD_LENGTH,DEFENDED,1.0,158,44.82,3.0,2.0,INSWINGER


In [8]:
# Apply the validated metric definitions
df_features = add_metric_features(df)

print("Raw dataset shape:", df.shape)
print("Feature dataset shape:", df_features.shape)

new_columns = sorted(set(df_features.columns) - set(df.columns))
print("\nDerived metric columns:", new_columns)

Raw dataset shape: (34967, 61)
Feature dataset shape: (34967, 68)

Derived metric columns: ['batting_dot', 'boundary_ball', 'boundary_runs', 'bowler_credited_wicket', 'bowling_dot', 'legal_delivery', 'match_phase']


In [9]:
# Match-level datasets
match_summary = build_match_summary(df_features)
team_match = build_team_match(df_features)

# Player-level datasets
player_season_batting = build_player_season_batting(df_features)
player_season_bowling = build_player_season_bowling(df_features)

player_season_phase_batting = build_player_season_phase_batting(df_features)
player_season_phase_bowling = build_player_season_phase_bowling(df_features)

# Display dataset sizes
dataset_summary = pd.DataFrame([
    {"dataset": "match_summary", "rows": len(match_summary)},
    {"dataset": "team_match", "rows": len(team_match)},
    {"dataset": "player_season_batting", "rows": len(player_season_batting)},
    {"dataset": "player_season_bowling", "rows": len(player_season_bowling)},
    {"dataset": "player_season_phase_batting", "rows": len(player_season_phase_batting)},
    {"dataset": "player_season_phase_bowling", "rows": len(player_season_phase_bowling)},
])

display(dataset_summary)

,dataset,rows
0,match_summary,145
1,team_match,290
2,player_season_batting,347
3,player_season_bowling,273
4,player_season_phase_batting,733
5,player_season_phase_bowling,672


In [10]:
assert df_features.shape[0] == df.shape[0]
assert len(match_summary) == 145
assert len(team_match) == 290

assert not player_season_batting.duplicated(["p_bat", "year"]).any()
assert not player_season_bowling.duplicated(["p_bowl", "year"]).any()

assert not player_season_phase_batting.duplicated(
    ["p_bat", "year", "match_phase"]
).any()

assert not player_season_phase_bowling.duplicated(
    ["p_bowl", "year", "match_phase"]
).any()

print("All analytical input checks passed.")

All analytical input checks passed.


In [11]:
# Filter KKR's team-match records
kkr_matches = team_match.loc[
    team_match["team"].eq("Kolkata Knight Riders")
].copy()

print("KKR team-match records:", len(kkr_matches))

display(
    kkr_matches.sort_values(["year", "date"])
    if "date" in kkr_matches.columns
    else kkr_matches.sort_values(["year", "p_match"])
)

KKR team-match records: 28


,p_match,team,opponent,winner,team_result,year
2,1359476,Kolkata Knight Riders,Punjab Kings,Punjab Kings,loss,2023
16,1359483,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,win,2023
25,1359487,Kolkata Knight Riders,Gujarat Titans,Kolkata Knight Riders,win,2023
36,1359493,Kolkata Knight Riders,Sunrisers Hyderabad,Sunrisers Hyderabad,loss,2023
42,1359496,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians,loss,2023
55,1359502,Kolkata Knight Riders,Delhi Capitals,Delhi Capitals,loss,2023
65,1359507,Kolkata Knight Riders,Chennai Super Kings,Chennai Super Kings,loss,2023
70,1359510,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,win,2023
77,1359513,Kolkata Knight Riders,Gujarat Titans,Gujarat Titans,loss,2023
92,1359521,Kolkata Knight Riders,Sunrisers Hyderabad,Kolkata Knight Riders,win,2023


In [12]:
print("Available columns:")
print(kkr_matches.columns.tolist())

display(kkr_matches.head())

Available columns:
['p_match', 'team', 'opponent', 'winner', 'team_result', 'year']


,p_match,team,opponent,winner,team_result,year
2,1359476,Kolkata Knight Riders,Punjab Kings,Punjab Kings,loss,2023
16,1359483,Kolkata Knight Riders,Royal Challengers Bengaluru,Kolkata Knight Riders,win,2023
25,1359487,Kolkata Knight Riders,Gujarat Titans,Kolkata Knight Riders,win,2023
36,1359493,Kolkata Knight Riders,Sunrisers Hyderabad,Sunrisers Hyderabad,loss,2023
42,1359496,Kolkata Knight Riders,Mumbai Indians,Mumbai Indians,loss,2023


In [14]:
# Summarise KKR results by season
kkr_results = (
    kkr_matches
    .groupby("year")
    .agg(
        matches_represented=("p_match", "nunique"),
        wins=("team_result", lambda s: s.eq("win").sum()),
        losses=("team_result", lambda s: s.eq("loss").sum()),
        unclassified=("team_result", lambda s: (~s.isin(["win", "loss"])).sum()),
    )
    .reset_index()
)

kkr_results["classified_matches"] = (
    kkr_results["wins"] + kkr_results["losses"]
)

kkr_results["win_pct"] = (
    100 * kkr_results["wins"]
    / kkr_results["classified_matches"].replace(0, pd.NA)
)

display(kkr_results.round(2))

,year,matches_represented,wins,losses,unclassified,classified_matches,win_pct
0,2023,14,6,8,0,14,42.86
1,2024,14,11,3,0,14,78.57


In [15]:
print("KKR result classifications:")
display(
    kkr_matches
    .groupby(["year", "team_result"], dropna=False)
    .size()
    .rename("matches")
    .reset_index()
)

print("Total represented matches:", kkr_matches["p_match"].nunique())
print("Unique KKR team-match records:", len(kkr_matches))

assert not kkr_matches["p_match"].duplicated().any()
assert kkr_matches["team_result"].isin(
    ["win", "loss", "unclassified"]
).all()

print("Match outcome checks passed.")

KKR result classifications:


,year,team_result,matches
0,2023,loss,8
1,2023,win,6
2,2024,loss,3
3,2024,win,11


Total represented matches: 28
Unique KKR team-match records: 28
Match outcome checks passed.


In [17]:
print(
    df_features[
        ["p_match", "year", "inns", "team_bat", "score"]
    ].head()
)

   p_match  year  inns             team_bat  score
0  1359475  2023     1  Chennai Super Kings      0
1  1359475  2023     1  Chennai Super Kings      1
2  1359475  2023     1  Chennai Super Kings      0
3  1359475  2023     1  Chennai Super Kings      1
4  1359475  2023     1  Chennai Super Kings      0


In [18]:
# Aggregate delivery scores to team-innings totals
team_innings = (
    df_features
    .groupby(
        ["p_match", "year", "inns", "team_bat"],
        as_index=False,
        dropna=False,
    )
    .agg(
        innings_score=("score", "sum"),
        recorded_deliveries=("score", "size"),
    )
)

# Keep KKR batting innings
kkr_innings = team_innings.loc[
    team_innings["team_bat"].eq("Kolkata Knight Riders")
].copy()

display(kkr_innings.head())
print("KKR innings records:", len(kkr_innings))

,p_match,year,inns,team_bat,innings_score,recorded_deliveries
3,1359476,2023,2,Kolkata Knight Riders,146,99
16,1359483,2023,1,Kolkata Knight Riders,204,129
25,1359487,2023,2,Kolkata Knight Riders,207,123
37,1359493,2023,2,Kolkata Knight Riders,204,128
42,1359496,2023,1,Kolkata Knight Riders,185,128


KKR innings records: 28


In [19]:
# Summarise KKR's innings scoring by season
kkr_scoring_summary = (
    kkr_innings
    .groupby("year")
    .agg(
        innings=("p_match", "size"),
        mean_score=("innings_score", "mean"),
        median_score=("innings_score", "median"),
        std_score=("innings_score", "std"),
        min_score=("innings_score", "min"),
        max_score=("innings_score", "max"),
        innings_180_plus=(
            "innings_score",
            lambda s: s.ge(180).sum(),
        ),
        innings_200_plus=(
            "innings_score",
            lambda s: s.ge(200).sum(),
        ),
    )
    .reset_index()
)

display(kkr_scoring_summary.round(2))

,year,innings,mean_score,median_score,std_score,min_score,max_score,innings_180_plus,innings_200_plus
0,2023,14,175.86,180.5,25.16,127,207,7,4
1,2024,14,190.14,177.5,47.01,114,272,7,6


In [20]:
display(
    kkr_innings[
        ["year", "p_match", "inns", "innings_score"]
    ]
    .sort_values(["year", "innings_score"], ascending=[True, False])
    .reset_index(drop=True)
)

,year,p_match,inns,innings_score
0,2023,1359487,2,207
1,2023,1359483,1,204
2,2023,1359493,2,204
3,2023,1359510,1,200
4,2023,1359507,2,186
5,2023,1359496,1,185
6,2023,1359527,2,182
7,2023,1359513,1,179
8,2023,1359542,2,175
9,2023,1359521,1,171


In [21]:
print(match_summary.columns.tolist())
display(match_summary.head())

['p_match', 'year', 'match_date', 'ground', 'country', 'winner', 'toss', 'represented_batting_teams', 'represented_bowling_teams', 'represented_teams', 'represented_innings', 'single_batting_team_flag', 'metadata_conflict_flag', 'result_class']


,p_match,year,match_date,ground,country,winner,toss,represented_batting_teams,represented_bowling_teams,represented_teams,represented_innings,single_batting_team_flag,metadata_conflict_flag,result_class
0,1359475,2023,3/31/2023,"Narendra Modi Stadium, Ahmedabad",India,Gujarat Titans,Gujarat Titans,2,2,2,2,False,False,classified
1,1359476,2023,4/1/2023,"Punjab Cricket Association IS Bindra Stadium, ...",India,Punjab Kings,Kolkata Knight Riders,2,2,2,2,False,False,classified
2,1359477,2023,4/1/2023,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,India,Lucknow Super Giants,Delhi Capitals,2,2,2,2,False,False,classified
3,1359478,2023,4/2/2023,"Rajiv Gandhi International Stadium, Uppal, Hyd...",India,Rajasthan Royals,Sunrisers Hyderabad,2,2,2,2,False,False,classified
4,1359479,2023,4/2/2023,"M Chinnaswamy Stadium, Bengaluru",India,Royal Challengers Bengaluru,Royal Challengers Bangalore,2,2,2,2,False,False,classified


In [22]:
# Count represented innings for each match
match_innings_coverage = (
    team_innings
    .groupby(["p_match", "year"])
    .agg(
        represented_innings=("inns", "nunique"),
        represented_batting_teams=("team_bat", "nunique"),
    )
    .reset_index()
)

display(
    match_innings_coverage
    .groupby(["year", "represented_innings"])
    .size()
    .rename("matches")
    .reset_index()
)

,year,represented_innings,matches
0,2023,1,1
1,2023,2,73
2,2024,2,71


In [23]:
incomplete_innings_coverage = match_innings_coverage.loc[
    match_innings_coverage["represented_innings"].ne(2)
]

display(incomplete_innings_coverage)

,p_match,year,represented_innings,represented_batting_teams
44,1359519,2023,1,1


In [24]:
# Retain matches with exactly two represented innings
complete_innings_matches = match_innings_coverage.loc[
    match_innings_coverage["represented_innings"].eq(2),
    ["p_match", "year"],
]

# Keep the innings totals from those matches
complete_team_innings = team_innings.merge(
    complete_innings_matches,
    on=["p_match", "year"],
    how="inner",
    validate="many_to_one",
)

print("Matches retained:", complete_innings_matches["p_match"].nunique())
print("Team-innings records retained:", len(complete_team_innings))

Matches retained: 144
Team-innings records retained: 288


In [25]:
# KKR's innings
kkr_innings_complete = complete_team_innings.loc[
    complete_team_innings["team_bat"].eq("Kolkata Knight Riders"),
    ["p_match", "year", "inns", "innings_score"],
].rename(
    columns={
        "inns": "kkr_innings_number",
        "innings_score": "kkr_score",
    }
)

# Opponents' innings
opponent_innings = complete_team_innings.loc[
    ~complete_team_innings["team_bat"].eq("Kolkata Knight Riders"),
    ["p_match", "year", "inns", "team_bat", "innings_score"],
].rename(
    columns={
        "inns": "opponent_innings_number",
        "team_bat": "opponent",
        "innings_score": "opponent_score",
    }
)

# Match-level KKR scorecard
kkr_scorecard = kkr_innings_complete.merge(
    opponent_innings,
    on=["p_match", "year"],
    how="inner",
    validate="one_to_one",
).merge(
    kkr_matches[["p_match", "year", "team_result"]],
    on=["p_match", "year"],
    how="inner",
    validate="one_to_one",
)

kkr_scorecard["kkr_batted_first"] = (
    kkr_scorecard["kkr_innings_number"].eq(1)
)

display(
    kkr_scorecard.sort_values(["year", "p_match"])
)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge

In [26]:
# Check whether KKR has more than one represented innings per match
kkr_duplicate_keys = (
    kkr_innings_complete
    .groupby(["p_match", "year"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

# Check whether the opponent innings table has multiple rows per match
opponent_duplicate_keys = (
    opponent_innings
    .groupby(["p_match", "year"])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

print("Duplicate KKR match keys:", len(kkr_duplicate_keys))
print("Duplicate opponent match keys:", len(opponent_duplicate_keys))

display(opponent_duplicate_keys)

if not opponent_duplicate_keys.empty:
    display(
        opponent_innings.merge(
            opponent_duplicate_keys[["p_match", "year"]],
            on=["p_match", "year"],
            how="inner",
        ).sort_values(["year", "p_match", "opponent_innings_number"])
    )

Duplicate KKR match keys: 0
Duplicate opponent match keys: 116


,p_match,year,row_count
0,1359475,2023,2
2,1359477,2023,2
3,1359478,2023,2
4,1359479,2023,2
5,1359480,2023,2
...,...,...,...
137,1426305,2024,2
138,1426306,2024,2
139,1426307,2024,2
141,1426310,2024,2


,p_match,year,opponent_innings_number,opponent,opponent_score
0,1359475,2023,1,Chennai Super Kings,178
1,1359475,2023,2,Gujarat Titans,182
2,1359477,2023,1,Lucknow Super Giants,193
3,1359477,2023,2,Delhi Capitals,143
4,1359478,2023,1,Rajasthan Royals,203
...,...,...,...,...,...
227,1426307,2024,2,Sunrisers Hyderabad,215
228,1426310,2024,1,Royal Challengers Bengaluru,172
229,1426310,2024,2,Rajasthan Royals,174
230,1426311,2024,1,Sunrisers Hyderabad,175


In [27]:
# 1. Get KKR's innings from completed matches
kkr_innings_complete = complete_team_innings.loc[
    complete_team_innings["team_bat"].eq("Kolkata Knight Riders"),
    ["p_match", "year", "inns", "innings_score"],
].rename(
    columns={
        "inns": "kkr_innings_number",
        "innings_score": "kkr_score",
    }
)

# 2. Identify the match IDs in which KKR played
kkr_match_keys = kkr_innings_complete[["p_match", "year"]].drop_duplicates()

# 3. Select only the non-KKR innings from those same matches
opponent_innings = (
    complete_team_innings.merge(
        kkr_match_keys,
        on=["p_match", "year"],
        how="inner",
        validate="many_to_one",
    )
    .loc[
        lambda x: ~x["team_bat"].eq("Kolkata Knight Riders"),
        ["p_match", "year", "inns", "team_bat", "innings_score"],
    ]
    .rename(
        columns={
            "inns": "opponent_innings_number",
            "team_bat": "opponent",
            "innings_score": "opponent_score",
        }
    )
)

# 4. Build the match-level KKR scorecard
kkr_scorecard = (
    kkr_innings_complete.merge(
        opponent_innings,
        on=["p_match", "year"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        kkr_matches[["p_match", "year", "team_result"]],
        on=["p_match", "year"],
        how="inner",
        validate="one_to_one",
    )
)

# 5. Determine whether KKR batted first
kkr_scorecard["kkr_batted_first"] = (
    kkr_scorecard["kkr_innings_number"].eq(1)
)

print("KKR completed matches:", len(kkr_scorecard))
print("Duplicate match keys:", kkr_scorecard.duplicated(
    ["p_match", "year"]
).sum())

display(
    kkr_scorecard.sort_values(["year", "p_match"])
)

KKR completed matches: 28
Duplicate match keys: 0


,p_match,year,kkr_innings_number,kkr_score,opponent_innings_number,opponent,opponent_score,team_result,kkr_batted_first
0,1359476,2023,2,146,1,Punjab Kings,191,loss,False
1,1359483,2023,1,204,2,Royal Challengers Bengaluru,123,win,True
2,1359487,2023,2,207,1,Gujarat Titans,204,win,False
3,1359493,2023,2,204,1,Sunrisers Hyderabad,228,loss,False
4,1359496,2023,1,185,2,Mumbai Indians,186,loss,True
5,1359502,2023,1,127,2,Delhi Capitals,128,loss,True
6,1359507,2023,2,186,1,Chennai Super Kings,235,loss,False
7,1359510,2023,1,200,2,Royal Challengers Bengaluru,179,win,True
8,1359513,2023,1,179,2,Gujarat Titans,180,loss,True
9,1359521,2023,1,171,2,Sunrisers Hyderabad,166,win,True


In [28]:
kkr_batting_order_summary = (
    kkr_scorecard
    .groupby(
        ["year", "kkr_batted_first"],
        as_index=False,
        observed=True,
    )
    .agg(
        matches=("p_match", "nunique"),
        wins=("team_result", lambda s: s.eq("win").sum()),
        losses=("team_result", lambda s: s.eq("loss").sum()),
    )
)

kkr_batting_order_summary["win_pct"] = (
    100
    * kkr_batting_order_summary["wins"]
    / kkr_batting_order_summary["matches"]
)

kkr_batting_order_summary["batting_order"] = (
    kkr_batting_order_summary["kkr_batted_first"]
    .map({
        True: "Batted first",
        False: "Chased",
    })
)

kkr_batting_order_summary = kkr_batting_order_summary[
    [
        "year",
        "batting_order",
        "matches",
        "wins",
        "losses",
        "win_pct",
    ]
].sort_values(["year", "batting_order"])

display(kkr_batting_order_summary.round({"win_pct": 2}))

,year,batting_order,matches,wins,losses,win_pct
1,2023,Batted first,7,3,4,42.86
0,2023,Chased,7,3,4,42.86
3,2024,Batted first,9,6,3,66.67
2,2024,Chased,5,5,0,100.00


In [29]:
kkr_batting_order_scores = (
    kkr_scorecard
    .groupby(
        ["year", "kkr_batted_first"],
        as_index=False,
        observed=True,
    )
    .agg(
        matches=("p_match", "nunique"),
        mean_kkr_score=("kkr_score", "mean"),
        mean_opponent_score=("opponent_score", "mean"),
        mean_score_difference=(
            "kkr_score",
            lambda s: (
                s - kkr_scorecard.loc[s.index, "opponent_score"]
            ).mean(),
        ),
    )
)

kkr_batting_order_scores["batting_order"] = (
    kkr_batting_order_scores["kkr_batted_first"]
    .map({
        True: "Batted first",
        False: "Chased",
    })
)

display(
    kkr_batting_order_scores[
        [
            "year",
            "batting_order",
            "matches",
            "mean_kkr_score",
            "mean_opponent_score",
            "mean_score_difference",
        ]
    ].round(2)
)

,year,batting_order,matches,mean_kkr_score,mean_opponent_score,mean_score_difference
0,2023,Chased,7,178.14,193.86,-15.71
1,2023,Batted first,7,173.57,159.00,14.57
2,2024,Chased,5,156.40,153.60,2.80
3,2024,Batted first,9,208.89,181.22,27.67


In [30]:
kkr_successful_chases_2024 = (
    kkr_scorecard.loc[
        kkr_scorecard["year"].eq(2024)
        & ~kkr_scorecard["kkr_batted_first"]
        & kkr_scorecard["team_result"].eq("win"),
        [
            "p_match",
            "opponent",
            "opponent_score",
            "kkr_score",
            "team_result",
        ],
    ]
    .copy()
    .sort_values("opponent_score", ascending=False)
)

kkr_successful_chases_2024["runs_above_target"] = (
    kkr_successful_chases_2024["kkr_score"]
    - kkr_successful_chases_2024["opponent_score"]
)

display(kkr_successful_chases_2024)

,p_match,opponent,opponent_score,kkr_score,team_result,runs_above_target
15,1422128,Royal Challengers Bengaluru,182,186,win,4
18,1426266,Lucknow Super Giants,161,162,win,1
26,1426309,Sunrisers Hyderabad,159,164,win,5
22,1426285,Delhi Capitals,153,156,win,3
27,1426312,Sunrisers Hyderabad,113,114,win,1


In [31]:
kkr_deliveries = df_features.loc[
    df_features["team_bat"].eq("Kolkata Knight Riders")
].copy()

kkr_phase_summary = (
    kkr_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        batting_runs=("batruns", "sum"),
        balls_faced=("ballfaced", "sum"),
        batting_dots=("batting_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs=("boundary_runs", "sum"),
        innings_count=("p_match", lambda s: s.nunique()),
    )
    .reset_index()
)

kkr_phase_summary["strike_rate"] = (
    100
    * kkr_phase_summary["batting_runs"]
    / kkr_phase_summary["balls_faced"].replace(0, pd.NA)
)

kkr_phase_summary["dot_ball_pct"] = (
    100
    * kkr_phase_summary["batting_dots"]
    / kkr_phase_summary["balls_faced"].replace(0, pd.NA)
)

kkr_phase_summary["boundary_ball_pct"] = (
    100
    * kkr_phase_summary["boundary_balls"]
    / kkr_phase_summary["balls_faced"].replace(0, pd.NA)
)

kkr_phase_summary["boundary_run_pct"] = (
    100
    * kkr_phase_summary["boundary_runs"]
    / kkr_phase_summary["batting_runs"].replace(0, pd.NA)
)

phase_order = ["Powerplay", "Middle", "Death"]

kkr_phase_summary["match_phase"] = pd.Categorical(
    kkr_phase_summary["match_phase"],
    categories=phase_order,
    ordered=True,
)

display(
    kkr_phase_summary.sort_values(["year", "match_phase"]).round(2)
)

,year,match_phase,batting_runs,balls_faced,batting_dots,boundary_balls,boundary_runs,innings_count,strike_rate,dot_ball_pct,boundary_ball_pct,boundary_run_pct
2,2023,Powerplay,651,507,264,109,494,14,128.40,52.07,21.50,75.88
1,2023,Middle,1065,760,233,130,626,14,140.13,30.66,17.11,58.78
0,2023,Death,622,390,146,88,438,14,159.49,37.44,22.56,70.42
5,2024,Powerplay,849,507,231,153,702,14,167.46,45.56,30.18,82.69
4,2024,Middle,1114,725,224,149,712,14,153.66,30.90,20.55,63.91
3,2024,Death,518,275,83,77,384,12,188.36,30.18,28.00,74.13


In [32]:
kkr_phase_coverage = (
    kkr_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        represented_matches=("p_match", "nunique"),
        represented_innings=("p_match", lambda s: (
            kkr_deliveries.loc[s.index, ["p_match", "inns"]]
            .drop_duplicates()
            .shape[0]
        )),
        delivery_records=("p_match", "size"),
        legal_balls=("legal_delivery", "sum"),
        balls_faced=("ballfaced", "sum"),
    )
    .reset_index()
)

kkr_phase_coverage["match_phase"] = pd.Categorical(
    kkr_phase_coverage["match_phase"],
    categories=["Powerplay", "Middle", "Death"],
    ordered=True,
)

display(
    kkr_phase_coverage
    .sort_values(["year", "match_phase"])
)

,year,match_phase,represented_matches,represented_innings,delivery_records,legal_balls,balls_faced
2,2023,Powerplay,14,14,529,504,507
1,2023,Middle,14,14,781,756,760
0,2023,Death,14,14,418,387,390
5,2024,Powerplay,14,14,531,504,507
4,2024,Middle,14,14,759,721,725
3,2024,Death,12,12,304,269,275


In [33]:
# Distinct KKR innings represented in the Death phase
death_innings = (
    kkr_deliveries.loc[
        kkr_deliveries["match_phase"].eq("Death")
    ]
    [["year", "p_match", "inns"]]
    .drop_duplicates()
)

# All represented KKR innings
all_kkr_innings = (
    kkr_deliveries
    [["year", "p_match", "inns"]]
    .drop_duplicates()
)

# Identify innings with no Death-phase records
innings_without_death = (
    all_kkr_innings
    .merge(
        death_innings.assign(has_death_phase=True),
        on=["year", "p_match", "inns"],
        how="left",
        validate="one_to_one",
    )
    .loc[lambda x: x["has_death_phase"].isna()]
    .drop(columns="has_death_phase")
)

display(innings_without_death.sort_values(["year", "p_match"]))

,year,p_match,inns
26,2024,1426309,2
27,2024,1426312,2


In [34]:
# Inspect the full innings context for the identified cases
innings_context = (
    kkr_deliveries
    .groupby(["year", "p_match", "inns"], as_index=False)
    .agg(
        innings_score=("score", "sum"),
        last_over=("over_num", "max"),
        delivery_records=("score", "size"),
        legal_balls=("legal_delivery", "sum"),
        balls_faced=("ballfaced", "sum"),
    )
)

display(
    innings_context.merge(
        innings_without_death,
        on=["year", "p_match", "inns"],
        how="inner",
        validate="one_to_one",
    )
)

,year,p_match,inns,innings_score,last_over,delivery_records,legal_balls,balls_faced
0,2024,1426309,2,164,14,84,82,82
1,2024,1426312,2,114,11,67,63,63


In [35]:
kkr_powerplay_batters = (
    kkr_deliveries.loc[
        kkr_deliveries["match_phase"].eq("Powerplay")
    ]
    .groupby(["year", "p_bat", "bat"], as_index=False)
    .agg(
        runs=("batruns", "sum"),
        balls_faced=("ballfaced", "sum"),
        dot_balls=("batting_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        matches=("p_match", "nunique"),
    )
)

kkr_powerplay_batters["strike_rate"] = (
    100
    * kkr_powerplay_batters["runs"]
    / kkr_powerplay_batters["balls_faced"].replace(0, pd.NA)
)

kkr_powerplay_batters["dot_ball_pct"] = (
    100
    * kkr_powerplay_batters["dot_balls"]
    / kkr_powerplay_batters["balls_faced"].replace(0, pd.NA)
)

kkr_powerplay_batters["boundary_ball_pct"] = (
    100
    * kkr_powerplay_batters["boundary_balls"]
    / kkr_powerplay_batters["balls_faced"].replace(0, pd.NA)
)

display(
    kkr_powerplay_batters
    .sort_values(["year", "runs"], ascending=[True, False])
    .round(2)
)

,year,p_bat,bat,runs,balls_faced,dot_balls,boundary_balls,matches,strike_rate,dot_ball_pct,boundary_ball_pct
1,2023,298438,Jason Roy,177,121,59,33,7,146.28,48.76,27.27
8,2023,974087,Rahmanullah Gurbaz,145,112,60,22,10,129.46,53.57,19.64
7,2023,851403,Venkatesh Iyer,142,109,55,23,12,130.28,50.46,21.10
5,2023,604527,Nitish Rana,94,71,35,15,12,132.39,49.30,21.13
9,2023,1048813,Narayan Jagadeesan,63,58,32,11,6,108.62,55.17,18.97
6,2023,723105,Rinku Singh,18,11,4,3,2,163.64,36.36,27.27
2,2023,398506,Mandeep Singh,4,8,4,0,3,50.00,50.00,0.00
4,2023,536936,Litton Das,4,4,3,1,1,100.00,75.00,25.00
10,2023,1079839,Anukul Roy,4,5,4,1,1,80.00,80.00,20.00
0,2023,230558,Sunil Narine,0,4,4,0,2,0.00,100.00,0.00


In [36]:
# Total KKR Powerplay runs by season
kkr_powerplay_totals = (
    kkr_powerplay_batters
    .groupby("year", as_index=False)
    .agg(
        team_powerplay_runs=("runs", "sum"),
        team_powerplay_balls=("balls_faced", "sum"),
    )
)

# Calculate each batter's share of team Powerplay runs
kkr_powerplay_contributions = kkr_powerplay_batters.merge(
    kkr_powerplay_totals,
    on="year",
    how="left",
    validate="many_to_one",
)

kkr_powerplay_contributions["run_share_pct"] = (
    100
    * kkr_powerplay_contributions["runs"]
    / kkr_powerplay_contributions["team_powerplay_runs"]
)

# Identify whether the player appeared in both seasons
player_season_presence = (
    kkr_powerplay_batters
    .groupby("p_bat")["year"]
    .nunique()
    .rename("seasons_present")
    .reset_index()
)

kkr_powerplay_contributions = kkr_powerplay_contributions.merge(
    player_season_presence,
    on="p_bat",
    how="left",
    validate="many_to_one",
)

kkr_powerplay_contributions["presence_category"] = (
    kkr_powerplay_contributions["seasons_present"]
    .map({
        1: "One season only",
        2: "Both seasons",
    })
)

display(
    kkr_powerplay_contributions[
        [
            "year",
            "bat",
            "runs",
            "balls_faced",
            "strike_rate",
            "run_share_pct",
            "presence_category",
        ]
    ]
    .sort_values(["year", "runs"], ascending=[True, False])
    .round(2)
)

,year,bat,runs,balls_faced,strike_rate,run_share_pct,presence_category
1,2023,Jason Roy,177,121,146.28,27.19,One season only
8,2023,Rahmanullah Gurbaz,145,112,129.46,22.27,Both seasons
7,2023,Venkatesh Iyer,142,109,130.28,21.81,Both seasons
5,2023,Nitish Rana,94,71,132.39,14.44,Both seasons
9,2023,Narayan Jagadeesan,63,58,108.62,9.68,One season only
6,2023,Rinku Singh,18,11,163.64,2.76,Both seasons
2,2023,Mandeep Singh,4,8,50.00,0.61,One season only
4,2023,Litton Das,4,4,100.00,0.61,One season only
10,2023,Anukul Roy,4,5,80.00,0.61,One season only
0,2023,Sunil Narine,0,4,0.00,0.00,Both seasons


In [37]:
# Keep only players represented in both seasons
returning_batters = kkr_powerplay_contributions.loc[
    kkr_powerplay_contributions["presence_category"].eq("Both seasons")
].copy()

returning_batter_comparison = returning_batters.pivot(
    index="bat",
    columns="year",
    values=["runs", "balls_faced", "strike_rate", "run_share_pct"],
)

# Flatten the multi-level column names
returning_batter_comparison.columns = [
    f"{metric}_{year}"
    for metric, year in returning_batter_comparison.columns
]

returning_batter_comparison = returning_batter_comparison.reset_index()

# Calculate changes where both seasons have observations
returning_batter_comparison["runs_change"] = (
    returning_batter_comparison["runs_2024"]
    - returning_batter_comparison["runs_2023"]
)

returning_batter_comparison["balls_change"] = (
    returning_batter_comparison["balls_faced_2024"]
    - returning_batter_comparison["balls_faced_2023"]
)

returning_batter_comparison["strike_rate_change"] = (
    returning_batter_comparison["strike_rate_2024"]
    - returning_batter_comparison["strike_rate_2023"]
)

display(
    returning_batter_comparison.sort_values(
        "runs_change",
        ascending=False,
    ).round(2)
)

,bat,runs_2023,runs_2024,balls_faced_2023,balls_faced_2024,strike_rate_2023,strike_rate_2024,run_share_pct_2023,run_share_pct_2024,runs_change,balls_change,strike_rate_change
3,Sunil Narine,0.0,281.0,4.0,165.0,0.00,170.30,0.00,33.10,281.0,161.0,170.30
2,Rinku Singh,18.0,9.0,11.0,7.0,163.64,128.57,2.76,1.06,-9.0,-4.0,-35.06
4,Venkatesh Iyer,142.0,111.0,109.0,50.0,130.28,222.00,21.81,13.07,-31.0,-59.0,91.72
0,Nitish Rana,94.0,12.0,71.0,14.0,132.39,85.71,14.44,1.41,-82.0,-57.0,-46.68
1,Rahmanullah Gurbaz,145.0,44.0,112.0,36.0,129.46,122.22,22.27,5.18,-101.0,-76.0,-7.24


In [38]:
# Filter deliveries where KKR was the bowling team
kkr_bowling_deliveries = df_features.loc[
    df_features["team_bowl"].eq("Kolkata Knight Riders")
].copy()

# Aggregate bowling performance by season and phase
kkr_bowling_phase = (
    kkr_bowling_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        bowler_credited_wickets=("bowler_credited_wicket", "sum"),
        delivery_records=("p_match", "size"),
        represented_matches=("p_match", "nunique"),
    )
    .reset_index()
)

# Convert legal balls into overs and calculate bowling rates
kkr_bowling_phase["overs_bowled"] = (
    kkr_bowling_phase["legal_balls"] / 6
)

kkr_bowling_phase["runs_per_over"] = (
    kkr_bowling_phase["runs_conceded"]
    / kkr_bowling_phase["overs_bowled"].replace(0, pd.NA)
)

kkr_bowling_phase["dot_ball_pct"] = (
    100
    * kkr_bowling_phase["dot_balls"]
    / kkr_bowling_phase["legal_balls"].replace(0, pd.NA)
)

kkr_bowling_phase["wickets_per_20_overs"] = (
    20
    * kkr_bowling_phase["bowler_credited_wickets"]
    / kkr_bowling_phase["overs_bowled"].replace(0, pd.NA)
)

# Sort phases in cricket order
phase_order = ["Powerplay", "Middle", "Death"]

kkr_bowling_phase["match_phase"] = pd.Categorical(
    kkr_bowling_phase["match_phase"],
    categories=phase_order,
    ordered=True,
)

display(
    kkr_bowling_phase
    .sort_values(["year", "match_phase"])
    .round(2)
)

,year,match_phase,runs_conceded,legal_balls,dot_balls,bowler_credited_wickets,delivery_records,represented_matches,overs_bowled,runs_per_over,dot_ball_pct,wickets_per_20_overs
2,2023,Powerplay,806,504,198,20,525,14,84.00,9.60,39.29,4.76
1,2023,Middle,1009,745,247,36,758,14,124.17,8.13,33.15,5.80
0,2023,Death,609,345,86,21,374,13,57.50,10.59,24.93,7.30
5,2024,Powerplay,817,504,211,27,540,14,84.00,9.73,41.87,6.43
4,2024,Middle,1022,756,265,49,783,14,126.00,8.11,35.05,7.78
3,2024,Death,517,316,121,28,328,14,52.67,9.82,38.29,10.63


In [39]:
kkr_bowling_coverage = (
    kkr_bowling_deliveries
    .groupby(["year", "match_phase"], observed=True)
    .agg(
        represented_matches=("p_match", "nunique"),
        represented_innings=("p_match", lambda s: (
            kkr_bowling_deliveries.loc[
                s.index, ["p_match", "inns"]
            ]
            .drop_duplicates()
            .shape[0]
        )),
        delivery_records=("p_match", "size"),
        legal_balls=("legal_delivery", "sum"),
    )
    .reset_index()
)

kkr_bowling_coverage["match_phase"] = pd.Categorical(
    kkr_bowling_coverage["match_phase"],
    categories=["Powerplay", "Middle", "Death"],
    ordered=True,
)

display(
    kkr_bowling_coverage
    .sort_values(["year", "match_phase"])
)

,year,match_phase,represented_matches,represented_innings,delivery_records,legal_balls
2,2023,Powerplay,14,14,525,504
1,2023,Middle,14,14,758,745
0,2023,Death,13,13,374,345
5,2024,Powerplay,14,14,540,504
4,2024,Middle,14,14,783,756
3,2024,Death,14,14,328,316


In [40]:
kkr_death_bowlers = (
    kkr_bowling_deliveries.loc[
        kkr_bowling_deliveries["match_phase"].eq("Death")
    ]
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        matches=("p_match", "nunique"),
    )
)

kkr_death_bowlers["overs_bowled"] = (
    kkr_death_bowlers["legal_balls"] / 6
)

kkr_death_bowlers["runs_per_over"] = (
    kkr_death_bowlers["runs_conceded"]
    / kkr_death_bowlers["overs_bowled"].replace(0, pd.NA)
)

kkr_death_bowlers["dot_ball_pct"] = (
    100
    * kkr_death_bowlers["dot_balls"]
    / kkr_death_bowlers["legal_balls"].replace(0, pd.NA)
)

kkr_death_bowlers["wickets_per_20_overs"] = (
    20
    * kkr_death_bowlers["wickets"]
    / kkr_death_bowlers["overs_bowled"].replace(0, pd.NA)
)

display(
    kkr_death_bowlers
    .sort_values(["year", "wickets"], ascending=[True, False])
    .round(2)
)

,year,p_bowl,bowl,runs_conceded,legal_balls,dot_balls,wickets,matches,overs_bowled,runs_per_over,dot_ball_pct,wickets_per_20_overs
9,2023,1108375,Varun Chakravarthy,106,76,24,4,11,12.67,8.37,31.58,6.32
0,2023,230558,Sunil Narine,69,48,12,3,8,8.00,8.62,25.00,7.50
4,2023,475281,Shardul Thakur,88,47,11,3,6,7.83,11.23,23.40,7.66
10,2023,1209292,Vaibhav Arora,67,36,8,3,5,6.00,11.17,22.22,10.00
2,2023,276298,Andre Russell,60,31,8,2,5,5.17,11.61,25.81,7.74
8,2023,1083033,Kulwant Khejroliya,37,14,4,2,2,2.33,15.86,28.57,17.14
5,2023,493773,Lockie Ferguson,37,16,1,1,2,2.67,13.88,6.25,7.50
6,2023,604527,Nitish Rana,22,17,4,1,2,2.83,7.76,23.53,7.06
7,2023,1079839,Anukul Roy,6,6,2,1,1,1.00,6.00,33.33,20.00
12,2023,1350792,Suyash Sharma,68,36,9,1,6,6.00,11.33,25.00,3.33


In [41]:
death_bowling_contribution = kkr_death_bowlers.copy()

# Team totals for each season
death_team_totals = (
    death_bowling_contribution
    .groupby("year", as_index=False)
    .agg(
        team_legal_balls=("legal_balls", "sum"),
        team_wickets=("wickets", "sum"),
        team_runs_conceded=("runs_conceded", "sum"),
    )
)

death_bowling_contribution = death_bowling_contribution.merge(
    death_team_totals,
    on="year",
    how="left",
    validate="many_to_one",
)

death_bowling_contribution["legal_ball_share_pct"] = (
    100
    * death_bowling_contribution["legal_balls"]
    / death_bowling_contribution["team_legal_balls"]
)

death_bowling_contribution["wicket_share_pct"] = (
    100
    * death_bowling_contribution["wickets"]
    / death_bowling_contribution["team_wickets"]
)

display(
    death_bowling_contribution[
        [
            "year",
            "bowl",
            "matches",
            "legal_balls",
            "legal_ball_share_pct",
            "runs_per_over",
            "dot_ball_pct",
            "wickets",
            "wicket_share_pct",
        ]
    ]
    .sort_values(
        ["year", "legal_balls"],
        ascending=[True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

,year,bowl,matches,legal_balls,legal_ball_share_pct,runs_per_over,dot_ball_pct,wickets,wicket_share_pct
0,2023,Varun Chakravarthy,11,76,22.03,8.37,31.58,4,19.05
1,2023,Sunil Narine,8,48,13.91,8.62,25.00,3,14.29
2,2023,Shardul Thakur,6,47,13.62,11.23,23.40,3,14.29
3,2023,Vaibhav Arora,5,36,10.43,11.17,22.22,3,14.29
4,2023,Suyash Sharma,6,36,10.43,11.33,25.00,1,4.76
5,2023,Andre Russell,5,31,8.99,11.61,25.81,2,9.52
6,2023,Nitish Rana,2,17,4.93,7.76,23.53,1,4.76
7,2023,Lockie Ferguson,2,16,4.64,13.88,6.25,1,4.76
8,2023,Kulwant Khejroliya,2,14,4.06,15.86,28.57,2,9.52
9,2023,Tim Southee,1,6,1.74,15.00,16.67,0,0.00


In [42]:
kkr_middle_bowlers = (
    kkr_bowling_deliveries.loc[
        kkr_bowling_deliveries["match_phase"].eq("Middle")
    ]
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        matches=("p_match", "nunique"),
    )
)

kkr_middle_bowlers["overs_bowled"] = (
    kkr_middle_bowlers["legal_balls"] / 6
)

kkr_middle_bowlers["runs_per_over"] = (
    kkr_middle_bowlers["runs_conceded"]
    / kkr_middle_bowlers["overs_bowled"].replace(0, pd.NA)
)

kkr_middle_bowlers["dot_ball_pct"] = (
    100
    * kkr_middle_bowlers["dot_balls"]
    / kkr_middle_bowlers["legal_balls"].replace(0, pd.NA)
)

kkr_middle_bowlers["wickets_per_20_overs"] = (
    20
    * kkr_middle_bowlers["wickets"]
    / kkr_middle_bowlers["overs_bowled"].replace(0, pd.NA)
)

display(
    kkr_middle_bowlers
    .sort_values(
        ["year", "legal_balls"],
        ascending=[True, False],
    )
    .round(2)
    .reset_index(drop=True)
)

,year,p_bowl,bowl,runs_conceded,legal_balls,dot_balls,wickets,matches,overs_bowled,runs_per_over,dot_ball_pct,wickets_per_20_overs
0,2023,230558,Sunil Narine,245,198,52,6,14,33.00,7.42,26.26,3.64
1,2023,1350792,Suyash Sharma,231,180,64,6,11,30.00,7.70,35.56,4.00
2,2023,1108375,Varun Chakravarthy,190,144,52,11,14,24.00,7.92,36.11,9.17
3,2023,475281,Shardul Thakur,82,55,17,3,7,9.17,8.95,30.91,6.55
4,2023,1079839,Anukul Roy,59,42,18,2,3,7.00,8.43,42.86,5.71
5,2023,604527,Nitish Rana,27,30,14,2,4,5.00,5.40,46.67,8.00
6,2023,276298,Andre Russell,39,24,8,1,3,4.00,9.75,33.33,5.00
7,2023,376116,Umesh Yadav,53,24,4,1,3,4.00,13.25,16.67,5.00
8,2023,493773,Lockie Ferguson,39,18,8,0,3,3.00,13.00,44.44,0.00
9,2023,1312645,Harshit Rana,15,12,3,1,2,2.00,7.50,25.00,10.00


In [43]:
from pathlib import Path

metric_file = project_root / "src" / "features" / "metric_definitions.py"
test_file = project_root / "tests" / "test_metric_definitions.py"

print("=== metric_definitions.py ===")
print(metric_file.read_text())

print("\n=== test_metric_definitions.py ===")
print(test_file.read_text())

=== metric_definitions.py ===
"""Reusable metric definitions for delivery-level IPL analytics.

The functions in this module derive analytical features without modifying
the input DataFrame.

Important:
- The raw dataset must remain unchanged.
- Batting dot balls remain a provisional definition.
- Source bowlruns anomalies must not be silently corrected.
"""

from __future__ import annotations
from unittest import result

import pandas as pd


# Dismissals included in the candidate bowler-credited wicket metric.
BOWLER_CREDITED_DISMISSALS = {
    "caught",
    "bowled",
    "leg before wicket",
    "stumped",
    "hit wicket",
}


def derive_legal_delivery(df: pd.DataFrame) -> pd.Series:
    """Return whether each delivery is legal.

    A delivery is legal when neither a wide nor a no-ball is recorded.
    """
    required_columns = {"wide", "noball"}
    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing colum

In [44]:
from pathlib import Path

player_level_file = project_root / "src" / "features" / "player_level.py"

print(player_level_file.read_text())

"""Build player-level analytical datasets from delivery records."""

from __future__ import annotations

import pandas as pd


# Grouping keys defining each analytical dataset.
PLAYER_SEASON_BATTING_KEYS = [
    "p_bat",
    "year",
]

PLAYER_SEASON_BOWLING_KEYS = [
    "p_bowl",
    "year",
]

PLAYER_SEASON_PHASE_BATTING_KEYS = [
    "p_bat",
    "year",
    "match_phase",
]

PLAYER_SEASON_PHASE_BOWLING_KEYS = [
    "p_bowl",
    "year",
    "match_phase",
]


def build_player_season_batting(
        
    df: pd.DataFrame,
) -> pd.DataFrame:
    """Build one row per batter per season."""

    required_columns = {
        "p_bat",
        "bat",
        "year",
        "batruns",
        "ballfaced",
        "batting_dot",
        "boundary_ball",
        "boundary_runs",
    }

    missing_columns = required_columns - set(df.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    # Aggregate the unde

In [45]:
df_features = add_metric_features(df)

player_season_bowling = build_player_season_bowling(df_features)

player_season_phase_bowling = build_player_season_phase_bowling(
    df_features
)

display(player_season_bowling.head())
display(player_season_phase_bowling.head())

,p_bowl,year,bowler_name,runs_conceded,legal_balls,dot_balls,wickets,economy_rate,bowling_dot_pct,wickets_per_100_balls
0,8917,2023,Moeen Ali,195,156,54,9,7.500000,34.615385,5.769231
1,8917,2024,Moeen Ali,71,48,12,2,8.875000,25.000000,4.166667
2,25913,2024,Mohammad Nabi,109,74,18,2,8.837838,24.324324,2.702703
3,26421,2023,Ravichandran Ashwin,368,294,88,14,7.510204,29.931973,4.761905
4,26421,2024,Ravichandran Ashwin,467,330,87,9,8.490909,26.363636,2.727273


,p_bowl,year,match_phase,bowler_name,runs_conceded,legal_balls,dot_balls,wickets,economy_rate,bowling_dot_pct,wickets_per_100_balls
0,8917,2023,Death,Moeen Ali,3,6,3,1,3.000000,50.000000,16.666667
1,8917,2023,Middle,Moeen Ali,182,138,45,6,7.913043,32.608696,4.347826
2,8917,2023,Powerplay,Moeen Ali,10,12,6,2,5.000000,50.000000,16.666667
3,8917,2024,Death,Moeen Ali,19,12,4,1,9.500000,33.333333,8.333333
4,8917,2024,Middle,Moeen Ali,52,36,8,1,8.666667,22.222222,2.777778


In [49]:
import importlib

import src.features.metric_definitions as metric_definitions
import src.features.player_level as player_level

# Reload metric definitions first.
importlib.reload(metric_definitions)

# Reload player-level aggregations.
importlib.reload(player_level)

# Refresh the function references used in the notebook.
add_metric_features = metric_definitions.add_metric_features

build_player_season_bowling = player_level.build_player_season_bowling
build_player_season_phase_bowling = (
    player_level.build_player_season_phase_bowling
)

In [51]:
# Rebuild delivery-level features with the new extras metrics.
df_features = add_metric_features(df)

# Rebuild both bowling aggregations.
player_season_bowling = build_player_season_bowling(df_features)

player_season_phase_bowling = (
    build_player_season_phase_bowling(df_features)
)

print("Player-season bowling shape:", player_season_bowling.shape)
print("Player-season-phase bowling shape:", player_season_phase_bowling.shape)

print("\nPlayer-season bowling columns:")
print(player_season_bowling.columns.tolist())

print("\nPlayer-season-phase bowling columns:")
print(player_season_phase_bowling.columns.tolist())

Player-season bowling shape: (273, 18)
Player-season-phase bowling shape: (672, 19)

Player-season bowling columns:
['p_bowl', 'year', 'bowler_name', 'runs_conceded', 'legal_balls', 'dot_balls', 'wickets', 'boundary_balls', 'boundary_runs_conceded', 'wides_conceded', 'noball_extras_conceded', 'wide_noball_extras_conceded', 'overs_bowled', 'economy_rate', 'bowling_dot_pct', 'wickets_per_100_balls', 'boundary_balls_per_100_legal_balls', 'extras_per_20_overs']

Player-season-phase bowling columns:
['p_bowl', 'year', 'match_phase', 'bowler_name', 'runs_conceded', 'legal_balls', 'dot_balls', 'wickets', 'boundary_balls', 'boundary_runs_conceded', 'wides_conceded', 'noball_extras_conceded', 'wide_noball_extras_conceded', 'overs_bowled', 'economy_rate', 'bowling_dot_pct', 'wickets_per_100_balls', 'boundary_balls_per_100_legal_balls', 'extras_per_20_overs']


In [52]:
new_bowling_metrics = [
    "boundary_runs_conceded",
    "boundary_balls",
    "boundary_balls_per_100_legal_balls",
    "wides_conceded",
    "noball_extras_conceded",
    "wide_noball_extras_conceded",
    "extras_per_20_overs",
]

for name, table in [
    ("Player-season bowling", player_season_bowling),
    ("Player-season-phase bowling", player_season_phase_bowling),
]:
    missing = sorted(set(new_bowling_metrics) - set(table.columns))

    print(f"{name}:")
    print("Missing metrics:", missing)

    assert not missing, f"{name} is missing metrics: {missing}"

print("\nAll new bowling metrics are present.")

Player-season bowling:
Missing metrics: []
Player-season-phase bowling:
Missing metrics: []

All new bowling metrics are present.


In [53]:
import pandas as pd

# Expected totals directly from the delivery-level feature dataset.
expected_totals = {
    "boundary_balls": df_features["boundary_ball"].sum(),
    "boundary_runs_conceded": df_features["boundary_runs"].sum(),
    "wides_conceded": df_features["wides"].sum(),
    "noball_extras_conceded": df_features["noball_extras"].sum(),
    "wide_noball_extras_conceded": (
        df_features["wide_noball_extras"].sum()
    ),
}

# Actual totals across player-season bowling records.
actual_totals = {
    metric: player_season_bowling[metric].sum()
    for metric in expected_totals
}

reconciliation = pd.DataFrame({
    "expected_delivery_total": expected_totals,
    "actual_aggregation_total": actual_totals,
})

reconciliation["difference"] = (
    reconciliation["actual_aggregation_total"]
    - reconciliation["expected_delivery_total"]
)

display(reconciliation)

assert (reconciliation["difference"] == 0).all(), (
    "Bowling metric totals do not reconcile to delivery-level data."
)

print("All five bowling metric totals reconcile.")

,expected_delivery_total,actual_aggregation_total,difference
boundary_balls,6735,6735,0
boundary_runs_conceded,31710,31710,0
wides_conceded,1591,1591,0
noball_extras_conceded,175,175,0
wide_noball_extras_conceded,1766,1766,0


All five bowling metric totals reconcile.


In [54]:
# Recalculate rates independently from aggregated counts.
valid_legal_balls = player_season_bowling["legal_balls"].where(
    player_season_bowling["legal_balls"] != 0
)

valid_overs = player_season_bowling["overs_bowled"].where(
    player_season_bowling["overs_bowled"] != 0
)

expected_boundary_rate = (
    player_season_bowling["boundary_balls"]
    .div(valid_legal_balls)
    .mul(100)
)

expected_extras_rate = (
    player_season_bowling["wide_noball_extras_conceded"]
    .div(valid_overs)
    .mul(20)
)

assert (
    player_season_bowling["boundary_balls_per_100_legal_balls"]
    .sub(expected_boundary_rate)
    .abs()
    .fillna(0)
    .lt(1e-10)
    .all()
)

assert (
    player_season_bowling["extras_per_20_overs"]
    .sub(expected_extras_rate)
    .abs()
    .fillna(0)
    .lt(1e-10)
    .all()
)

print("Both rate calculations are valid.")

Both rate calculations are valid.


In [55]:
existing_bowling_metrics = [
    "runs_conceded",
    "legal_balls",
    "dot_balls",
    "wickets",
    "economy_rate",
    "bowling_dot_pct",
    "wickets_per_100_balls",
]

print(
    player_season_bowling[
        ["p_bowl", "year"] + existing_bowling_metrics
    ].head(10)
)

print("\nPlayer-season-phase bowling preview:")

print(
    player_season_phase_bowling[
        ["p_bowl", "year", "match_phase"]
        + existing_bowling_metrics
    ].head(10)
)

   p_bowl  year  runs_conceded  legal_balls  dot_balls  wickets  economy_rate  \
0    8917  2023            195          156         54        9      7.500000   
1    8917  2024             71           48         12        2      8.875000   
2   25913  2024            109           74         18        2      8.837838   
3   26421  2023            368          294         88       14      7.510204   
4   26421  2024            467          330         87        9      8.490909   
5   30288  2023            223          129         37       10     10.372093   
6   30288  2024            254          144         39        7     10.583333   
7   31107  2023            149          114         30        7      7.842105   
8   31107  2024             20           12          5        1     10.000000   
9   32966  2023            495          366        120       22      8.114754   

   bowling_dot_pct  wickets_per_100_balls  
0        34.615385               5.769231  
1        25.000000  

In [56]:
# Filter deliveries where KKR was bowling.
kkr_bowling_deliveries = df_features.loc[
    df_features["team_bowl"].eq("Kolkata Knight Riders")
].copy()

# Aggregate underlying counts by season and match phase.
kkr_bowling_phase = (
    kkr_bowling_deliveries
    .groupby(["year", "match_phase"], as_index=False)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs_conceded=("boundary_runs", "sum"),
        wides_conceded=("wides", "sum"),
        noball_extras_conceded=("noball_extras", "sum"),
        wide_noball_extras_conceded=(
            "wide_noball_extras", "sum"
        ),
        delivery_records=("p_match", "size"),
        represented_matches=("p_match", "nunique"),
    )
)

# Calculate rates from the aggregated counts.
kkr_bowling_phase["overs_bowled"] = (
    kkr_bowling_phase["legal_balls"] / 6
)

valid_balls = kkr_bowling_phase["legal_balls"].where(
    kkr_bowling_phase["legal_balls"] != 0
)

valid_overs = kkr_bowling_phase["overs_bowled"].where(
    kkr_bowling_phase["overs_bowled"] != 0
)

kkr_bowling_phase["runs_per_over"] = (
    kkr_bowling_phase["runs_conceded"]
    .div(valid_overs)
)

kkr_bowling_phase["dot_ball_pct"] = (
    kkr_bowling_phase["dot_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_bowling_phase["wickets_per_20_overs"] = (
    kkr_bowling_phase["wickets"]
    .div(valid_overs)
    .mul(20)
)

kkr_bowling_phase["boundary_balls_per_100_legal_balls"] = (
    kkr_bowling_phase["boundary_balls"]
    .div(valid_balls)
    .mul(100)
)

# Present extras as a rate per over as well as a total.
kkr_bowling_phase["extras_per_over"] = (
    kkr_bowling_phase["wide_noball_extras_conceded"]
    .div(valid_overs)
)

kkr_bowling_phase["boundary_runs_per_over"] = (
    kkr_bowling_phase["boundary_runs_conceded"]
    .div(valid_overs)
)

# Put phases in cricket order.
phase_order = ["Powerplay", "Middle", "Death"]

kkr_bowling_phase["match_phase"] = pd.Categorical(
    kkr_bowling_phase["match_phase"],
    categories=phase_order,
    ordered=True,
)

kkr_bowling_phase = kkr_bowling_phase.sort_values(
    ["match_phase", "year"]
).reset_index(drop=True)

display(
    kkr_bowling_phase[
        [
            "year",
            "match_phase",
            "represented_matches",
            "overs_bowled",
            "runs_conceded",
            "runs_per_over",
            "wickets",
            "wickets_per_20_overs",
            "dot_ball_pct",
            "boundary_balls",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_conceded",
            "boundary_runs_per_over",
            "wides_conceded",
            "noball_extras_conceded",
            "wide_noball_extras_conceded",
            "extras_per_over",
        ]
    ].round(2)
)

,year,match_phase,represented_matches,overs_bowled,runs_conceded,runs_per_over,wickets,wickets_per_20_overs,dot_ball_pct,boundary_balls,boundary_balls_per_100_legal_balls,boundary_runs_conceded,boundary_runs_per_over,wides_conceded,noball_extras_conceded,wide_noball_extras_conceded,extras_per_over
0,2023,Powerplay,14,84.00,806,9.60,20,4.76,39.29,132,26.19,584,6.95,24,6,30,0.36
1,2024,Powerplay,14,84.00,817,9.73,27,6.43,41.87,131,25.99,598,7.12,37,5,42,0.50
2,2023,Middle,14,124.17,1009,8.13,36,5.80,33.15,120,16.11,578,4.66,10,3,13,0.10
3,2024,Middle,14,126.00,1022,8.11,49,7.78,35.05,121,16.01,588,4.67,25,3,28,0.22
4,2023,Death,13,57.50,609,10.59,21,7.30,24.93,72,20.87,360,6.26,25,6,31,0.54
5,2024,Death,14,52.67,517,9.82,28,10.63,38.29,66,20.89,354,6.72,20,0,20,0.38


In [57]:
# Filter KKR bowling deliveries in the death phase.
kkr_death_deliveries = kkr_bowling_deliveries.loc[
    kkr_bowling_deliveries["match_phase"].eq("Death")
].copy()

# Aggregate each bowler's death-over performance by season.
kkr_death_bowler = (
    kkr_death_deliveries
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        legal_balls=("legal_delivery", "sum"),
        runs_conceded=("bowlruns", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        dot_balls=("bowling_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs_conceded=("boundary_runs", "sum"),
        wides_conceded=("wides", "sum"),
        noball_extras_conceded=("noball_extras", "sum"),
        wide_noball_extras_conceded=(
            "wide_noball_extras", "sum"
        ),
        matches=("p_match", "nunique"),
    )
)

# Calculate workload and performance rates.
kkr_death_bowler["overs_bowled"] = (
    kkr_death_bowler["legal_balls"] / 6
)

valid_balls = kkr_death_bowler["legal_balls"].where(
    kkr_death_bowler["legal_balls"] != 0
)

valid_overs = kkr_death_bowler["overs_bowled"].where(
    kkr_death_bowler["overs_bowled"] != 0
)

kkr_death_bowler["runs_per_over"] = (
    kkr_death_bowler["runs_conceded"]
    .div(valid_overs)
)

kkr_death_bowler["dot_ball_pct"] = (
    kkr_death_bowler["dot_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_death_bowler["wickets_per_20_overs"] = (
    kkr_death_bowler["wickets"]
    .div(valid_overs)
    .mul(20)
)

kkr_death_bowler["boundary_balls_per_100_legal_balls"] = (
    kkr_death_bowler["boundary_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_death_bowler["boundary_runs_per_over"] = (
    kkr_death_bowler["boundary_runs_conceded"]
    .div(valid_overs)
)

kkr_death_bowler["extras_per_over"] = (
    kkr_death_bowler["wide_noball_extras_conceded"]
    .div(valid_overs)
)

# Sort by season and workload to make the main contributors visible.
kkr_death_bowler = kkr_death_bowler.sort_values(
    ["year", "legal_balls"],
    ascending=[True, False],
).reset_index(drop=True)

display(
    kkr_death_bowler[
        [
            "year",
            "bowl",
            "matches",
            "overs_bowled",
            "runs_conceded",
            "runs_per_over",
            "wickets",
            "wickets_per_20_overs",
            "dot_ball_pct",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_per_over",
            "wides_conceded",
            "noball_extras_conceded",
            "extras_per_over",
        ]
    ].round(2)
)

,year,bowl,matches,overs_bowled,runs_conceded,runs_per_over,wickets,wickets_per_20_overs,dot_ball_pct,boundary_balls_per_100_legal_balls,boundary_runs_per_over,wides_conceded,noball_extras_conceded,extras_per_over
0,2023,Varun Chakravarthy,11,12.67,106,8.37,4,6.32,31.58,14.47,4.89,3,0,0.24
1,2023,Sunil Narine,8,8.00,69,8.62,3,7.50,25.00,14.58,4.00,2,0,0.25
2,2023,Shardul Thakur,6,7.83,88,11.23,3,7.66,23.40,25.53,7.66,3,0,0.38
3,2023,Vaibhav Arora,5,6.00,67,11.17,3,10.00,22.22,22.22,5.33,8,2,1.67
4,2023,Suyash Sharma,6,6.00,68,11.33,1,3.33,25.00,25.00,7.33,1,1,0.33
5,2023,Andre Russell,5,5.17,60,11.61,2,7.74,25.81,22.58,7.35,4,0,0.77
6,2023,Nitish Rana,2,2.83,22,7.76,1,7.06,23.53,11.76,3.53,1,0,0.35
7,2023,Lockie Ferguson,2,2.67,37,13.88,1,7.50,6.25,31.25,9.00,0,1,0.38
8,2023,Kulwant Khejroliya,2,2.33,37,15.86,2,17.14,28.57,28.57,9.43,0,2,0.86
9,2023,Tim Southee,1,1.00,15,15.00,0,0.00,16.67,33.33,10.00,0,0,0.00


In [58]:
# Filter KKR bowling deliveries in the middle phase.
kkr_middle_deliveries = kkr_bowling_deliveries.loc[
    kkr_bowling_deliveries["match_phase"].eq("Middle")
].copy()

# Aggregate performance by bowler and season.
kkr_middle_bowler = (
    kkr_middle_deliveries
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        legal_balls=("legal_delivery", "sum"),
        runs_conceded=("bowlruns", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        dot_balls=("bowling_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs_conceded=("boundary_runs", "sum"),
        wides_conceded=("wides", "sum"),
        noball_extras_conceded=("noball_extras", "sum"),
        wide_noball_extras_conceded=(
            "wide_noball_extras", "sum"
        ),
        matches=("p_match", "nunique"),
    )
)

# Calculate rates.
kkr_middle_bowler["overs_bowled"] = (
    kkr_middle_bowler["legal_balls"] / 6
)

valid_balls = kkr_middle_bowler["legal_balls"].where(
    kkr_middle_bowler["legal_balls"] != 0
)

valid_overs = kkr_middle_bowler["overs_bowled"].where(
    kkr_middle_bowler["overs_bowled"] != 0
)

kkr_middle_bowler["runs_per_over"] = (
    kkr_middle_bowler["runs_conceded"]
    .div(valid_overs)
)

kkr_middle_bowler["dot_ball_pct"] = (
    kkr_middle_bowler["dot_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_middle_bowler["wickets_per_20_overs"] = (
    kkr_middle_bowler["wickets"]
    .div(valid_overs)
    .mul(20)
)

kkr_middle_bowler["boundary_balls_per_100_legal_balls"] = (
    kkr_middle_bowler["boundary_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_middle_bowler["boundary_runs_per_over"] = (
    kkr_middle_bowler["boundary_runs_conceded"]
    .div(valid_overs)
)

kkr_middle_bowler["extras_per_over"] = (
    kkr_middle_bowler["wide_noball_extras_conceded"]
    .div(valid_overs)
)

# Sort by season, then workload.
kkr_middle_bowler = kkr_middle_bowler.sort_values(
    ["year", "legal_balls"],
    ascending=[True, False],
).reset_index(drop=True)

display(
    kkr_middle_bowler[
        [
            "year",
            "bowl",
            "matches",
            "overs_bowled",
            "runs_conceded",
            "runs_per_over",
            "wickets",
            "dot_ball_pct",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_per_over",
            "wides_conceded",
            "noball_extras_conceded",
            "extras_per_over",
        ]
    ].round(2)
)

,year,bowl,matches,overs_bowled,runs_conceded,runs_per_over,wickets,dot_ball_pct,boundary_balls_per_100_legal_balls,boundary_runs_per_over,wides_conceded,noball_extras_conceded,extras_per_over
0,2023,Sunil Narine,14,33.00,245,7.42,6,26.26,10.61,3.15,3,0,0.09
1,2023,Suyash Sharma,11,30.00,231,7.70,6,35.56,15.56,4.47,3,0,0.10
2,2023,Varun Chakravarthy,14,24.00,190,7.92,11,36.11,15.97,4.75,2,0,0.08
3,2023,Shardul Thakur,7,9.17,82,8.95,3,30.91,20.00,5.24,0,2,0.22
4,2023,Anukul Roy,3,7.00,59,8.43,2,42.86,19.05,6.00,1,0,0.14
5,2023,Nitish Rana,4,5.00,27,5.40,2,46.67,10.00,2.40,0,0,0.00
6,2023,Andre Russell,3,4.00,39,9.75,1,33.33,20.83,6.00,0,0,0.00
7,2023,Umesh Yadav,3,4.00,53,13.25,1,16.67,37.50,10.00,0,0,0.00
8,2023,Lockie Ferguson,3,3.00,39,13.00,0,44.44,38.89,11.33,0,1,0.33
9,2023,Harshit Rana,2,2.00,15,7.50,1,25.00,8.33,3.00,1,0,0.50


In [59]:
# Filter KKR bowling deliveries in the Powerplay.
kkr_powerplay_deliveries = kkr_bowling_deliveries.loc[
    kkr_bowling_deliveries["match_phase"].eq("Powerplay")
].copy()

# Aggregate by bowler and season.
kkr_powerplay_bowler = (
    kkr_powerplay_deliveries
    .groupby(["year", "p_bowl", "bowl"], as_index=False)
    .agg(
        legal_balls=("legal_delivery", "sum"),
        runs_conceded=("bowlruns", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        dot_balls=("bowling_dot", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs_conceded=("boundary_runs", "sum"),
        wides_conceded=("wides", "sum"),
        noball_extras_conceded=("noball_extras", "sum"),
        wide_noball_extras_conceded=(
            "wide_noball_extras", "sum"
        ),
        matches=("p_match", "nunique"),
    )
)

# Calculate rates.
kkr_powerplay_bowler["overs_bowled"] = (
    kkr_powerplay_bowler["legal_balls"] / 6
)

valid_balls = kkr_powerplay_bowler["legal_balls"].where(
    kkr_powerplay_bowler["legal_balls"] != 0
)

valid_overs = kkr_powerplay_bowler["overs_bowled"].where(
    kkr_powerplay_bowler["overs_bowled"] != 0
)

kkr_powerplay_bowler["runs_per_over"] = (
    kkr_powerplay_bowler["runs_conceded"]
    .div(valid_overs)
)

kkr_powerplay_bowler["dot_ball_pct"] = (
    kkr_powerplay_bowler["dot_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_powerplay_bowler["boundary_balls_per_100_legal_balls"] = (
    kkr_powerplay_bowler["boundary_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_powerplay_bowler["boundary_runs_per_over"] = (
    kkr_powerplay_bowler["boundary_runs_conceded"]
    .div(valid_overs)
)

kkr_powerplay_bowler["extras_per_over"] = (
    kkr_powerplay_bowler["wide_noball_extras_conceded"]
    .div(valid_overs)
)

# Sort by season and workload.
kkr_powerplay_bowler = kkr_powerplay_bowler.sort_values(
    ["year", "legal_balls"],
    ascending=[True, False],
).reset_index(drop=True)

display(
    kkr_powerplay_bowler[
        [
            "year",
            "bowl",
            "matches",
            "overs_bowled",
            "runs_conceded",
            "runs_per_over",
            "wickets",
            "dot_ball_pct",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_per_over",
            "wides_conceded",
            "noball_extras_conceded",
            "extras_per_over",
        ]
    ].round(2)
)

,year,bowl,matches,overs_bowled,runs_conceded,runs_per_over,wickets,dot_ball_pct,boundary_balls_per_100_legal_balls,boundary_runs_per_over,wides_conceded,noball_extras_conceded,extras_per_over
0,2023,Varun Chakravarthy,13,16.0,133,8.31,5,40.62,21.88,5.50,7,0,0.44
1,2023,Umesh Yadav,8,14.0,123,8.79,0,42.86,22.62,6.57,1,0,0.07
2,2023,Harshit Rana,6,14.0,111,7.93,4,47.62,22.62,6.00,4,1,0.36
3,2023,Vaibhav Arora,5,9.0,78,8.67,0,31.48,20.37,5.11,0,2,0.22
4,2023,Sunil Narine,6,7.0,69,9.86,2,47.62,28.57,7.71,4,0,0.57
5,2023,Andre Russell,5,6.0,73,12.17,4,41.67,38.89,9.67,1,3,0.67
6,2023,Tim Southee,2,4.0,51,12.75,1,37.50,37.50,10.50,0,0,0.00
7,2023,Shardul Thakur,4,4.0,50,12.50,1,16.67,37.50,9.50,1,0,0.25
8,2023,Suyash Sharma,2,3.0,22,7.33,3,38.89,16.67,4.67,0,0,0.00
9,2023,David Wiese,1,2.0,23,11.50,0,25.00,25.00,7.00,0,0,0.00


In [60]:
# Keep matches with both innings represented in the dataset.
complete_match_ids = set(
    match_summary.loc[
        match_summary["represented_innings"].eq(2),
        "p_match",
    ]
)

# Filter KKR's bowling deliveries to those matches.
kkr_bowling_complete = kkr_bowling_deliveries.loc[
    kkr_bowling_deliveries["p_match"].isin(complete_match_ids)
].copy()

# Aggregate bowling performance at match grain.
kkr_bowling_match = (
    kkr_bowling_complete
    .groupby(["p_match", "year"], as_index=False)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        boundary_balls=("boundary_ball", "sum"),
        boundary_runs_conceded=("boundary_runs", "sum"),
        wides_conceded=("wides", "sum"),
        noball_extras_conceded=("noball_extras", "sum"),
        combined_extras=("wide_noball_extras", "sum"),
    )
)

# Calculate match-level rates.
kkr_bowling_match["overs_bowled"] = (
    kkr_bowling_match["legal_balls"] / 6
)

valid_balls = kkr_bowling_match["legal_balls"].where(
    kkr_bowling_match["legal_balls"] != 0
)

valid_overs = kkr_bowling_match["overs_bowled"].where(
    kkr_bowling_match["overs_bowled"] != 0
)

kkr_bowling_match["runs_per_over"] = (
    kkr_bowling_match["runs_conceded"]
    .div(valid_overs)
)

kkr_bowling_match["dot_ball_pct"] = (
    kkr_bowling_match["dot_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_bowling_match["boundary_balls_per_100_legal_balls"] = (
    kkr_bowling_match["boundary_balls"]
    .div(valid_balls)
    .mul(100)
)

kkr_bowling_match["boundary_runs_per_over"] = (
    kkr_bowling_match["boundary_runs_conceded"]
    .div(valid_overs)
)

kkr_bowling_match["extras_per_over"] = (
    kkr_bowling_match["combined_extras"]
    .div(valid_overs)
)

# Attach KKR's match result.
kkr_bowling_match = kkr_bowling_match.merge(
    kkr_matches[["p_match", "team_result"]],
    on="p_match",
    how="left",
    validate="one_to_one",
)

print("Match-level dataset shape:", kkr_bowling_match.shape)
print("\nRecords by season:")
display(
    kkr_bowling_match.groupby("year").agg(
        matches=("p_match", "nunique"),
        mean_economy=("runs_per_over", "mean"),
        median_economy=("runs_per_over", "median"),
    ).round(2)
)

display(
    kkr_bowling_match[
        [
            "p_match",
            "year",
            "team_result",
            "overs_bowled",
            "runs_per_over",
            "wickets",
            "dot_ball_pct",
            "boundary_balls_per_100_legal_balls",
            "boundary_runs_per_over",
            "extras_per_over",
        ]
    ].sort_values(["year", "runs_per_over"])
)

Match-level dataset shape: (28, 18)

Records by season:


,matches,mean_economy,median_economy
year,,,
2023,14,9.17,9.15
2024,14,8.94,8.28


,p_match,year,team_result,overs_bowled,runs_per_over,wickets,dot_ball_pct,boundary_balls_per_100_legal_balls,boundary_runs_per_over,extras_per_over
5,1359502,2023,loss,19.333333,6.517241,6,43.103448,14.655172,3.517241,0.258621
1,1359483,2023,win,17.666667,6.679245,10,45.283019,15.094340,4.075472,0.056604
12,1359535,2023,win,20.000000,7.200000,6,33.333333,10.000000,2.900000,0.300000
9,1359521,2023,win,20.000000,7.950000,8,35.000000,16.666667,4.500000,0.350000
13,1359542,2023,loss,20.000000,8.550000,8,39.166667,18.333333,5.400000,0.350000
10,1359527,2023,win,20.000000,8.600000,7,40.833333,22.500000,6.000000,0.300000
7,1359510,2023,win,20.000000,8.800000,7,23.333333,15.833333,4.500000,0.150000
0,1359476,2023,loss,20.000000,9.500000,5,35.000000,21.666667,6.100000,0.150000
8,1359513,2023,loss,17.833333,9.588785,3,24.299065,21.495327,6.056075,0.168224
2,1359487,2023,win,20.000000,9.950000,4,24.166667,20.833333,5.700000,0.550000


In [61]:
# Summarise the distribution of match-level economy rates.
match_economy_summary = (
    kkr_bowling_match
    .groupby("year")
    .agg(
        matches=("p_match", "nunique"),
        mean_economy=("runs_per_over", "mean"),
        median_economy=("runs_per_over", "median"),
        std_economy=("runs_per_over", "std"),
        minimum_economy=("runs_per_over", "min"),
        maximum_economy=("runs_per_over", "max"),
    )
    .round(2)
)

display(match_economy_summary)

# Count matches below selected economy-rate thresholds.
threshold_rows = []

for year, season_data in kkr_bowling_match.groupby("year"):
    for threshold in [8, 9, 10]:
        count = season_data["runs_per_over"].lt(threshold).sum()
        total = season_data["p_match"].nunique()

        threshold_rows.append({
            "year": year,
            "threshold_runs_per_over": threshold,
            "matches_below_threshold": int(count),
            "total_matches": int(total),
            "percentage_of_matches": round(
                count / total * 100, 2
            ),
        })

threshold_summary = pd.DataFrame(threshold_rows)

display(
    threshold_summary.pivot(
        index="threshold_runs_per_over",
        columns="year",
        values="matches_below_threshold",
    )
)

display(threshold_summary)

,matches,mean_economy,median_economy,std_economy,minimum_economy,maximum_economy
year,,,,,,
2023,14,9.17,9.15,1.72,6.52,11.75
2024,14,8.94,8.28,2.01,5.73,13.93


year,2023,2024
threshold_runs_per_over,,
8,4,4
9,7,9
10,10,10


,year,threshold_runs_per_over,matches_below_threshold,total_matches,percentage_of_matches
0,2023,8,4,14,28.57
1,2023,9,7,14,50.00
2,2023,10,10,14,71.43
3,2024,8,4,14,28.57
4,2024,9,9,14,64.29
5,2024,10,10,14,71.43


In [62]:
# Identify KKR's two 2024 knockout matches.
knockout_match_ids = {1426309, 1426312}

kkr_bowling_match["match_type"] = "League"

kkr_bowling_match.loc[
    kkr_bowling_match["p_match"].isin(knockout_match_ids),
    "match_type",
] = "Knockout"

# Inspect the classification before comparing seasons.
display(
    kkr_bowling_match[
        ["p_match", "year", "team_result", "match_type"]
    ].sort_values(["year", "match_type", "p_match"])
)

print("\nMatch counts by season and match type:")

display(
    kkr_bowling_match
    .groupby(["year", "match_type"], observed=True)
    .agg(
        matches=("p_match", "nunique"),
        mean_economy=("runs_per_over", "mean"),
        median_economy=("runs_per_over", "median"),
        mean_dot_ball_pct=("dot_ball_pct", "mean"),
        mean_wickets=("wickets", "mean"),
    )
    .round(2)
)

,p_match,year,team_result,match_type
0,1359476,2023,loss,League
1,1359483,2023,win,League
2,1359487,2023,win,League
3,1359493,2023,loss,League
4,1359496,2023,loss,League
5,1359502,2023,loss,League
6,1359507,2023,loss,League
7,1359510,2023,win,League
8,1359513,2023,loss,League
9,1359521,2023,win,League



Match counts by season and match type:


matches  mean_economy  median_economy  mean_dot_ball_pct  \
year match_type                                                             
2023 League           14          9.17            9.15              33.44   
2024 Knockout          2          6.94            6.94              42.25   
     League           12          9.27            8.59              37.29   

                 mean_wickets  
year match_type                
2023 League              5.50  
2024 Knockout            9.50  
     League              7.08

In [63]:
kkr_league_match_comparison = (
    kkr_bowling_match.loc[
        kkr_bowling_match["match_type"].eq("League")
    ]
    .groupby("year")
    .agg(
        matches=("p_match", "nunique"),
        mean_economy=("runs_per_over", "mean"),
        median_economy=("runs_per_over", "median"),
        std_economy=("runs_per_over", "std"),
        mean_dot_ball_pct=("dot_ball_pct", "mean"),
        mean_wickets=("wickets", "mean"),
        mean_boundary_runs_per_over=(
            "boundary_runs_per_over", "mean"
        ),
        mean_extras_per_over=("extras_per_over", "mean"),
    )
    .round(2)
)

display(kkr_league_match_comparison)

,matches,mean_economy,median_economy,std_economy,mean_dot_ball_pct,mean_wickets,mean_boundary_runs_per_over,mean_extras_per_over
year,,,,,,,,
2023,14,9.17,9.15,1.72,33.44,5.50,5.81,0.27
2024,12,9.27,8.59,1.92,37.29,7.08,6.21,0.34


In [64]:
# Step 14.1: Inspect the most expensive KKR league bowling performances

league_match_economy = kkr_bowling_match.loc[
    kkr_bowling_match["match_type"].eq("League")
].copy()

for year in [2023, 2024]:
    print(f"\n{'=' * 20} {year} {'=' * 20}")

    display(
        league_match_economy.loc[
            league_match_economy["year"].eq(year)
        ]
        .sort_values("runs_per_over", ascending=False)
        .head(5)[
            [
                "p_match",
                "team_result",
                "runs_conceded",
                "legal_balls",
                "overs_bowled",
                "runs_per_over",
                "dot_ball_pct",
                "wickets",
                "boundary_runs_per_over",
                "extras_per_over",
            ]
        ]
        .round(2)
    )


==================== 2023 ====================


,p_match,team_result,runs_conceded,legal_balls,overs_bowled,runs_per_over,dot_ball_pct,wickets,boundary_runs_per_over,extras_per_over
6,1359507,loss,235,120,20.00,11.75,28.33,4,8.20,0.15
3,1359493,loss,228,120,20.00,11.40,25.00,4,7.50,0.60
11,1359530,loss,149,79,13.17,11.32,35.44,0,9.11,0.23
4,1359496,loss,186,106,17.67,10.53,35.85,5,7.81,0.23
2,1359487,win,199,120,20.00,9.95,24.17,4,5.70,0.55



==================== 2024 ====================


,p_match,team_result,runs_conceded,legal_balls,overs_bowled,runs_per_over,dot_ball_pct,wickets,boundary_runs_per_over,extras_per_over
21,1426280,loss,260,112,18.67,13.93,24.11,1,10.93,0.21
19,1426269,loss,223,120,20.00,11.15,32.50,7,7.60,0.75
20,1426274,win,216,120,20.00,10.80,36.67,9,8.30,0.20
14,1422121,win,203,120,20.00,10.15,35.00,7,6.90,0.30
15,1422128,win,181,120,20.00,9.05,33.33,5,5.50,0.15


In [67]:
# Step 17.1: Build a match-level bowling dataset by phase.
# Compare league matches only; exclude the two 2024 knockouts.

league_match_ids = set(
    kkr_bowling_match.loc[
        kkr_bowling_match["match_type"].eq("League"),
        "p_match",
    ]
)

kkr_league_deliveries = kkr_bowling_deliveries.loc[
    kkr_bowling_deliveries["p_match"].isin(league_match_ids)
].copy()

# Derive the phase explicitly so this cell does not depend on
# whether a previous notebook cell added a phase column.
kkr_league_deliveries["phase"] = derive_match_phase(
    kkr_league_deliveries
)

kkr_league_phase_match = (
    kkr_league_deliveries
    .groupby(["p_match", "year", "phase"], observed=True)
    .agg(
        runs_conceded=("bowlruns", "sum"),
        legal_balls=("legal_delivery", "sum"),
        dot_balls=("bowling_dot", "sum"),
        wickets=("bowler_credited_wicket", "sum"),
        boundary_runs=("boundary_runs", "sum"),
        wides=("wides", "sum"),
        noball_extras=("noball_extras", "sum"),
    )
    .reset_index()
)

kkr_league_phase_match["overs_bowled"] = (
    kkr_league_phase_match["legal_balls"] / 6
)

kkr_league_phase_match["runs_per_over"] = (
    kkr_league_phase_match["runs_conceded"]
    / kkr_league_phase_match["overs_bowled"]
)

kkr_league_phase_match["dot_ball_pct"] = (
    kkr_league_phase_match["dot_balls"]
    / kkr_league_phase_match["legal_balls"]
    * 100
)

kkr_league_phase_match["boundary_runs_per_over"] = (
    kkr_league_phase_match["boundary_runs"]
    / kkr_league_phase_match["overs_bowled"]
)

kkr_league_phase_match["extras_per_over"] = (
    kkr_league_phase_match["wides"]
    + kkr_league_phase_match["noball_extras"]
) / kkr_league_phase_match["overs_bowled"]

display(
    kkr_league_phase_match.sort_values(
        ["year", "phase", "runs_per_over"],
        ascending=[True, True, False],
    ).round(2)
)

,p_match,year,phase,runs_conceded,legal_balls,dot_balls,wickets,boundary_runs,wides,noball_extras,overs_bowled,runs_per_over,dot_ball_pct,boundary_runs_per_over,extras_per_over
24,1359513,2023,Death,47,17,1,0,34,2,0,2.83,16.59,5.88,12.00,0.71
18,1359507,2023,Death,75,30,7,2,54,2,1,5.00,15.00,23.33,10.80,0.60
9,1359493,2023,Death,71,30,3,1,50,3,0,5.00,14.20,10.00,10.00,0.60
6,1359487,2023,Death,70,30,2,1,46,0,1,5.00,14.00,6.67,9.20,0.20
38,1359542,2023,Death,56,30,8,3,36,5,0,5.00,11.20,26.67,7.20,1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,1426298,2024,Powerplay,54,36,17,0,42,2,1,6.00,9.00,47.22,7.00,0.50
52,1426260,2024,Powerplay,52,36,15,1,36,2,0,6.00,8.67,41.67,6.00,0.33
55,1426266,2024,Powerplay,49,36,17,2,34,2,1,6.00,8.17,47.22,5.67,0.50
49,1422134,2024,Powerplay,47,36,19,4,36,2,0,6.00,7.83,52.78,6.00,0.33


In [68]:
# Step 17.2: Compare phase-level consistency across league matches.

phase_consistency = (
    kkr_league_phase_match
    .groupby(["year", "phase"], observed=True)
    .agg(
        matches=("p_match", "nunique"),
        mean_economy=("runs_per_over", "mean"),
        median_economy=("runs_per_over", "median"),
        std_economy=("runs_per_over", "std"),
        q25_economy=(
            "runs_per_over",
            lambda s: s.quantile(0.25),
        ),
        q75_economy=(
            "runs_per_over",
            lambda s: s.quantile(0.75),
        ),
        mean_dot_ball_pct=("dot_ball_pct", "mean"),
        mean_wickets=("wickets", "mean"),
        mean_boundary_runs_per_over=(
            "boundary_runs_per_over", "mean"
        ),
        mean_extras_per_over=("extras_per_over", "mean"),
    )
    .reset_index()
    .round(2)
)

display(
    phase_consistency.sort_values(["phase", "year"])
)

,year,phase,matches,mean_economy,median_economy,std_economy,q25_economy,q75_economy,mean_dot_ball_pct,mean_wickets,mean_boundary_runs_per_over,mean_extras_per_over
0,2023,Death,13,10.57,10.40,3.56,8.00,14.00,24.85,1.62,6.29,0.52
3,2024,Death,12,9.32,9.40,4.60,6.20,11.11,42.28,2.00,6.48,0.36
1,2023,Middle,14,8.15,8.72,2.17,6.97,9.92,33.11,2.57,4.69,0.10
4,2024,Middle,12,8.27,8.00,2.00,7.06,8.86,34.57,3.42,4.83,0.21
2,2023,Powerplay,14,9.60,9.25,1.43,8.50,10.08,39.29,1.43,6.95,0.36
5,2024,Powerplay,12,10.19,9.67,2.26,8.54,11.25,40.51,1.67,7.67,0.47


In [69]:
# Step 18.1: Build one record per complete KKR league match.

import numpy as np

# Restrict analysis to complete league matches.
league_match_ids = set(
    kkr_bowling_match.loc[
        kkr_bowling_match["match_type"].eq("League"),
        "p_match",
    ]
)

complete_match_ids = set(
    match_summary.loc[
        match_summary["represented_innings"].eq(2),
        "p_match",
    ]
)

eligible_match_ids = league_match_ids & complete_match_ids

# --------------------------------------------------
# 1. KKR batting performance
# --------------------------------------------------

kkr_batting_match = (
    df_features.loc[
        df_features["p_match"].isin(eligible_match_ids)
        & df_features["team_bat"].eq("Kolkata Knight Riders")
    ]
    .groupby(["p_match", "year", "inns"], observed=True)
    .agg(
        batting_runs=("score", "sum"),
        batting_balls=("ballfaced", "sum"),
        batting_dots=("batting_dot", "sum"),
        batting_boundary_balls=("boundary_ball", "sum"),
        batting_boundary_runs=("boundary_runs", "sum"),
    )
    .reset_index()
)

kkr_batting_match["batting_strike_rate"] = (
    kkr_batting_match["batting_runs"]
    / kkr_batting_match["batting_balls"]
    * 100
)

kkr_batting_match["batting_runs_per_over"] = (
    kkr_batting_match["batting_runs"]
    / (kkr_batting_match["batting_balls"] / 6)
)

# Determine whether KKR batted first or chased.
kkr_batting_match["batting_order"] = np.where(
    kkr_batting_match["inns"].eq(1),
    "Batted first",
    "Chased",
)

# --------------------------------------------------
# 2. Match-level KKR bowling performance
# --------------------------------------------------

kkr_bowling_match_eligible = kkr_bowling_match.loc[
    kkr_bowling_match["p_match"].isin(eligible_match_ids)
].copy()

# --------------------------------------------------
# 3. Combine batting, bowling and match outcome
# --------------------------------------------------

kkr_match_analysis = (
    kkr_batting_match.merge(
        kkr_bowling_match_eligible[
            [
                "p_match",
                "runs_per_over",
                "dot_ball_pct",
                "wickets",
                "boundary_runs_per_over",
                "extras_per_over",
                "team_result",
            ]
        ],
        on="p_match",
        how="inner",
        validate="one_to_one",
    )
)

print("Analytical dataset shape:", kkr_match_analysis.shape)
print("Unique matches:", kkr_match_analysis["p_match"].nunique())
print("\nSeason, batting order and outcome counts:")

display(
    kkr_match_analysis.groupby(
        ["year", "batting_order", "team_result"],
        observed=True,
    )
    .agg(matches=("p_match", "nunique"))
    .reset_index()
)

display(
    kkr_match_analysis.sort_values(["year", "p_match"]).head(10)
)

Analytical dataset shape: (26, 17)
Unique matches: 26

Season, batting order and outcome counts:


,year,batting_order,team_result,matches
0,2023,Batted first,loss,4
1,2023,Batted first,win,3
2,2023,Chased,loss,4
3,2023,Chased,win,3
4,2024,Batted first,loss,3
5,2024,Batted first,win,6
6,2024,Chased,win,3


,p_match,year,inns,batting_runs,batting_balls,batting_dots,batting_boundary_balls,batting_boundary_runs,batting_strike_rate,batting_runs_per_over,batting_order,runs_per_over,dot_ball_pct,wickets,boundary_runs_per_over,extras_per_over,team_result
0,1359476,2023,2,146,98,37,20,94,148.979592,8.938776,Chased,9.500000,35.000000,5,6.100000,0.150000,loss
1,1359483,2023,1,204,123,49,27,126,165.853659,9.951220,Batted first,6.679245,45.283019,10,4.075472,0.056604,win
2,1359487,2023,2,207,120,43,30,150,172.500000,10.350000,Chased,9.950000,24.166667,4,5.700000,0.550000,win
3,1359493,2023,2,204,122,44,29,138,167.213115,10.032787,Chased,11.400000,25.000000,4,7.500000,0.600000,loss
4,1359496,2023,1,185,120,44,22,108,154.166667,9.250000,Batted first,10.528302,35.849057,5,7.811321,0.226415,loss
5,1359502,2023,1,127,120,67,16,76,105.833333,6.350000,Batted first,6.517241,43.103448,6,3.517241,0.258621,loss
6,1359507,2023,2,186,120,50,26,128,155.000000,9.300000,Chased,11.750000,28.333333,4,8.200000,0.150000,loss
7,1359510,2023,1,200,120,37,28,136,166.666667,10.000000,Batted first,8.800000,23.333333,7,4.500000,0.150000,win
8,1359513,2023,1,179,120,49,24,120,149.166667,8.950000,Batted first,9.588785,24.299065,3,6.056075,0.168224,loss
9,1359521,2023,1,171,122,47,22,100,140.163934,8.409836,Batted first,7.950000,35.000000,8,4.500000,0.350000,win


In [70]:
# Step 19: Compare KKR wins and losses by batting order.

comparison_metrics = [
    "batting_runs",
    "batting_runs_per_over",
    "batting_strike_rate",
    "batting_dots",
    "batting_boundary_runs",
    "runs_per_over",
    "dot_ball_pct",
    "wickets",
    "boundary_runs_per_over",
    "extras_per_over",
]

win_loss_comparison = (
    kkr_match_analysis
    .groupby(
        ["batting_order", "team_result"],
        observed=True,
    )
    .agg(
        matches=("p_match", "nunique"),
        **{
            f"mean_{metric}": (metric, "mean")
            for metric in comparison_metrics
        },
        **{
            f"median_{metric}": (metric, "median")
            for metric in [
                "batting_runs",
                "batting_runs_per_over",
                "runs_per_over",
                "wickets",
            ]
        },
    )
    .round(2)
)

display(win_loss_comparison)

matches  mean_batting_runs  \
batting_order team_result                               
Batted first  loss               7             180.14   
              win                9             203.78   
Chased        loss               4             177.75   
              win                6             173.33   

                           mean_batting_runs_per_over  \
batting_order team_result                               
Batted first  loss                               8.98   
              win                               10.31   
Chased        loss                               9.26   
              win                                9.63   

                           mean_batting_strike_rate  mean_batting_dots  \
batting_order team_result                                                
Batted first  loss                           149.59              49.14   
              win                            171.87              41.22   
Chased        loss                           154.26              44.50   
              win                            160.43              39.00   

                           mean_batting_boundary_runs  mean_runs_per_over  \
batting_order team_result                                                   
Batted first  loss                             114.00               10.13   
              win                              135.33                8.57   
Chased        loss                             117.50               10.30   
              win                              111.67                8.38   

                           mean_dot_ball_pct  mean_wickets  \
batting_order team_result                                    
Batted first  loss                     33.16          3.57   
              win                      37.98          8.67   
Chased        loss                     31.88          5.25   
              win                      35.69          6.33   

                           mean_boundary_runs_per_over  mean_extras_per_over  \
batting_order team_result                                                      
Batted first  loss                                7.08                  0.29   
              win                                 5.52                  0.28   
Chased        loss                                6.80                  0.31   
              win                                 4.92                  0.36   

                           median_batting_runs  median_batting_runs_per_over  \
batting_order team_result                                                      
Batted first  loss                       179.0                          8.95   
              win                        204.0                         10.00   
Chased        loss                       180.5                          9.12   
              win                        172.0                          9.69   

                           median_runs_per_over  median_wickets  
batting_order team_result                                        
Batted first  loss                        10.53             3.0  
              win                          8.41             9.0  
Chased        loss                        10.45             4.5  
              win                          8.30             6.5

In [71]:
# Step 20: Check whether the win/loss patterns hold by season.

season_outcome_comparison = (
    kkr_match_analysis
    .groupby(
        ["year", "batting_order", "team_result"],
        observed=True,
    )
    .agg(
        matches=("p_match", "nunique"),
        mean_batting_runs=("batting_runs", "mean"),
        mean_batting_runs_per_over=("batting_runs_per_over", "mean"),
        mean_bowling_economy=("runs_per_over", "mean"),
        mean_bowling_dot_pct=("dot_ball_pct", "mean"),
        mean_bowling_wickets=("wickets", "mean"),
        mean_boundary_runs_conceded_per_over=(
            "boundary_runs_per_over", "mean"
        ),
    )
    .reset_index()
    .round(2)
)

display(season_outcome_comparison)

,year,batting_order,team_result,matches,mean_batting_runs,mean_batting_runs_per_over,mean_bowling_economy,mean_bowling_dot_pct,mean_bowling_wickets,mean_boundary_runs_conceded_per_over
0,2023,Batted first,loss,4,160.00,8.00,9.49,34.67,3.50,6.62
1,2023,Batted first,win,3,191.67,9.45,7.81,34.54,8.33,4.36
2,2023,Chased,loss,4,177.75,9.26,10.30,31.88,5.25,6.80
3,2023,Chased,win,3,178.67,9.11,8.58,32.78,5.67,4.87
4,2024,Batted first,loss,3,207.00,10.28,10.98,31.13,3.67,7.69
5,2024,Batted first,win,6,209.83,10.74,8.96,39.70,8.83,6.10
6,2024,Chased,win,3,168.00,10.14,8.18,38.61,7.00,4.97


In [72]:
# Phase 5, Step 1: Inspect existing player-level analytical outputs.

player_dataframes = [
    "player_batting",
    "player_bowling",
    "player_phase_batting",
    "player_phase_bowling",
    "kkr_batting_deliveries",
    "kkr_bowling_deliveries",
]

for name in player_dataframes:
    if name in globals():
        obj = globals()[name]
        print(f"\n{name}: {obj.shape}")
        print(list(obj.columns))
    else:
        print(f"\n{name}: not found")


player_batting: not found

player_bowling: not found

player_phase_batting: not found

player_phase_bowling: not found

kkr_batting_deliveries: not found

kkr_bowling_deliveries: (3308, 71)
['row_id', 'p_match', 'inns', 'bat', 'p_bat', 'team_bat', 'bowl', 'p_bowl', 'team_bowl', 'ball', 'ball_id', 'outcome', 'score', 'out', 'dismissal', 'p_out', 'over_num', 'noball', 'wide', 'byes', 'legbyes', 'cur_bat_runs', 'cur_bat_bf', 'cur_bowl_ovr', 'cur_bowl_wkts', 'cur_bowl_runs', 'inns_runs', 'inns_wkts', 'inns_balls', 'inns_runs_rem', 'inns_balls_rem', 'inns_rr', 'inns_rrr', 'target', 'max_balls', 'match_date', 'year', 'ground', 'country', 'winner', 'toss', 'competition', 'bat_hand', 'bowl_style', 'bowl_kind', 'batruns', 'ballfaced', 'bowlruns', 'bat_out', 'wagonx', 'wagony', 'wagonzone', 'line', 'length', 'shot', 'control', 'predscore', 'wprob', 'bowl_speed_category', 'bowl_release_point', 'bowl_type', 'legal_delivery', 'batting_dot', 'bowling_dot', 'bowler_credited_wicket', 'match_phase', '